#### Generators
Generators are a simpler way to create iterators. They use the yield keyword to produce a series of values lazily, which means they generate values on the fly and do not store them in memory.

In [1]:
def square(n):
    for i in range(3):
        yield i**2

In [2]:
square(3)

<generator object square at 0x0000027E9C6B7B90>

In [3]:
for i in square(3):
    print(i)

0
1
4


In [4]:
a=square(3)
a

<generator object square at 0x0000027E9C870AD0>

In [5]:
next(a)

0

In [6]:
def my_generator():
    yield 1
    yield 2
    yield 3

In [7]:
gen=my_generator()
gen

<generator object my_generator at 0x0000027E9C6DF320>

In [8]:
next(gen)

1

In [9]:
for val in gen:
    print(val)

2
3


#### Practical Example: Reading Large Files
Generators are particularly useful for reading large files because they allow you to process one line at a time without loading the entire file into memory.

In [10]:
### Practical : Reading LArge Files

def read_large_file(file_path):
    with open(file_path,'r') as file:
        for line in file:
            yield line

In [11]:
file_path='large_file.txt'

for line in read_large_file(file_path):
    print(line.strip())

Smt. Droupadi Murmu was sworn in as the 15th President of India on 25 July, 2022. Previously, she was the Governor of Jharkhand from 2015 to 2021. She has devoted her life to empowering the downtrodden and the marginalised sections and deepening the democratic values.

Early Life and Education

Born in a Santhali tribal family on 20 June, 1958 at Uparbeda village, Mayurbhanj, Odisha, Smt. Murmuâ€™s early life was marked by hardships and struggle. On completion of primary education from the village school, she went to Bhubaneswar on her own initiative to continue her studies. She earned the degree of Bachelor of Arts from Ramadevi Womenâ€™s College, Bhubaneswar and became the first woman from her village to receive college education.

Professional Career

From 1979 to 1983, Smt. Murmu served as a Junior Assistant in the Irrigation and Power Department, Government of Odisha. Later, she served as an honorary teacher at Sri Aurobindo Integral Education Centre, Rairangpur, from 1994 to 1997

#### Conclusion
Iterators and generators are powerful tools in Python for creating and handling sequences of data efficiently. Iterators provide a way to access elements sequentially, while generators allow you to generate items on the fly, making them particularly useful for handling large datasets and infinite sequences. Understanding these concepts will enable you to write more efficient and memory-conscious Python programs.

---
# 🎯 Generators: Interview Mastery & Production-Grade Concepts
---
This section covers advanced generator architectures, delegating generators (`yield from`), coroutines (`.send()`), generator pipelines, batch data loaders, and internal state inspection.

### Key Interview Topics:
1. **Generator Expressions vs. List Comprehensions (Memory & Laziness)**
2. **Sub-generator Delegation with `yield from` (Recursive Flattening)**
3. **Coroutines & Bidirectional Data Flow (`.send()`, `.throw()`, `.close()`)**
4. **Generator Pipelines for High-Throughput Streaming Data**
5. **Interview Coding Problem: Chunking / Batching Generator for Machine Learning**
6. **Generator Internal State Inspection (`gi_frame`, `gi_suspended`)**

### 1. Generator Expressions vs. List Comprehensions
**Interview Question:** *What is the difference between `[x for x in data]` and `(x for x in data)`? When should you use which?*

- **List Comprehension `[...]`:** Eagerly computes and loads the entire list into RAM immediately.
- **Generator Expression `(...)`:** Returns a lazy generator object that yields one value at a time on demand.
- **Rule of thumb:** Use list comprehensions when you need indexing, len, or multiple passes. Use generator expressions when streaming or passing directly into reducers like `sum()`, `min()`, `max()`.

In [12]:
import sys

# List comprehension: Eager allocation
list_comp = [x ** 2 for x in range(1_000_000)]
print(f"List comprehension RAM: {sys.getsizeof(list_comp) / (1024**2):.2f} MB")

# Generator expression: Lazy evaluation
gen_exp = (x ** 2 for x in range(1_000_000))
print(f"Generator expression RAM: {sys.getsizeof(gen_exp)} bytes")

# Direct reduction without creating intermediate list in RAM:
total_sum = sum(x ** 2 for x in range(1000))
print("Sum of squares:", total_sum)

List comprehension RAM: 8.06 MB
Generator expression RAM: 200 bytes
Sum of squares: 332833500


### 2. Delegating Generators with `yield from`
**Interview Question:** *What does `yield from` do in Python, and why is it superior to an inner `for item in subgen: yield item` loop?*

- **`yield from iterable`:** Delegates generation directly to a sub-iterable or sub-generator.
- **Transparent 2-way channel:** Forwards `.send()`, `.throw()`, and `.close()` directly to the inner generator, and captures the inner generator's `return` value!
- **Best Use Case:** Flattening arbitrarily nested data structures.

In [13]:
# Flattening deeply nested structures with 'yield from'
def deep_flatten(nested_list):
    for item in nested_list:
        if isinstance(item, (list, tuple)):
            yield from deep_flatten(item)  # Recursive delegation
        else:
            yield item

nested_data = [1, [2, [3, 4], 5], [6, [7, [8, 9]]]]
print("Original nested structure:", nested_data)
print("Flattened via yield from:", list(deep_flatten(nested_data)))

Original nested structure: [1, [2, [3, 4], 5], [6, [7, [8, 9]]]]
Flattened via yield from: [1, 2, 3, 4, 5, 6, 7, 8, 9]


### 3. Bidirectional Coroutines: Sending Values via `.send()`
**Interview Question:** *Can you send data into a running Python generator? How does the `yield` statement receive values?*

- A generator can act as a **coroutine** using `received = yield value`.
- **Priming:** A newly created generator must be advanced to the first `yield` using `next(gen)` or `gen.send(None)` before sending data into it.

In [14]:
# Running Average Coroutine using .send()
def running_average():
    total = 0.0
    count = 0
    average = None
    
    while True:
        # yield 'average' to caller, and receive 'new_val' from .send()
        new_val = yield average
        if new_val is None:
            break
        total += new_val
        count += 1
        average = total / count

# Usage:
avg_coroutine = running_average()
next(avg_coroutine) # Prime the coroutine to the first yield!

print("Sent 10 -> Running Avg:", avg_coroutine.send(10))
print("Sent 20 -> Running Avg:", avg_coroutine.send(20))
print("Sent 30 -> Running Avg:", avg_coroutine.send(30))
avg_coroutine.close()

Sent 10 -> Running Avg: 10.0
Sent 20 -> Running Avg: 15.0
Sent 30 -> Running Avg: 20.0


### 4. Streaming Generator Pipelines (Zero Memory Footprint)
**Interview Question:** *How do you process a 50 GB log file on a machine with only 4 GB of RAM?*

- Chain multiple generator functions together in a **streaming pipeline**.
- Each record flows through the pipeline one item at a time without accumulating in RAM.

In [15]:
# Streaming Pipeline Example: Parse and Filter Log Records
simulated_logs = [
    "2026-10-09 10:00:01 INFO User:101 status=200 duration=45ms",
    "2026-10-09 10:00:02 ERROR User:102 status=500 duration=850ms",
    "2026-10-09 10:00:03 INFO User:103 status=200 duration=30ms",
    "2026-10-09 10:00:04 ERROR User:104 status=503 duration=1200ms"
]

# Stage 1: Yield log lines
def stream_logs(logs):
    for line in logs:
        yield line

# Stage 2: Filter error lines
def filter_errors(log_stream):
    for line in log_stream:
        if "ERROR" in line:
            yield line

# Stage 3: Extract duration integer
def extract_durations(error_stream):
    for line in error_stream:
        dur_str = line.split("duration=")[1].replace("ms", "")
        yield int(dur_str)

# Compose the pipeline:
pipeline = extract_durations(filter_errors(stream_logs(simulated_logs)))

print("Durations of failed requests (ms):", list(pipeline))

Durations of failed requests (ms): [850, 1200]


### 5. Interview Coding Problem: Batching / Chunking Generator
**Interview Question:** *Implement a generator function `batch_generator(iterable, batch_size)` that chunks an iterable into batches of size `N`. The final batch may have fewer than `N` elements.*

In [16]:
# Batching Generator for Machine Learning & ETL
import itertools

def batch_generator(iterable, batch_size):
    iterator = iter(iterable)
    while True:
        # Extract up to batch_size items lazily:
        batch = list(itertools.islice(iterator, batch_size))
        if not batch:
            break
        yield batch

# Test with 11 elements in batches of 4:
dataset = range(1, 12)
for batch_idx, batch in enumerate(batch_generator(dataset, batch_size=4), 1):
    print(f"Batch {batch_idx}: {batch}")

Batch 1: [1, 2, 3, 4]
Batch 2: [5, 6, 7, 8]
Batch 3: [9, 10, 11]


### 6. Generator Internals & State Inspection
**Interview Question:** *How does Python keep track of execution inside a generator?*

- Python generators store an execution context in a `frame` object (`gen.gi_frame`).
- It has an instruction pointer and local variable dictionary.
- States can be inspected using `inspect.getgeneratorstate()`:
  - `GEN_CREATED` (before first `next()`)
  - `GEN_RUNNING` (currently executing)
  - `GEN_SUSPENDED` (paused at `yield`)
  - `GEN_CLOSED` (finished or closed)

In [17]:
import inspect

def demo_gen():
    yield "A"
    yield "B"

g = demo_gen()
print("State before start: ", inspect.getgeneratorstate(g))

next(g)
print("State at first yield:", inspect.getgeneratorstate(g))

list(g) # Exhaust remaining items
print("State after finish:  ", inspect.getgeneratorstate(g))

State before start:  GEN_CREATED
State at first yield: GEN_SUSPENDED
State after finish:   GEN_CLOSED
